# 02 · Retrieval baseline

Pipeline: SQLite documents → Voyage embeddings → Postgres/pgvector → semantic search → Voyage rerank → temporal decay.

Run top to bottom after a kernel restart. No state from notebook 01 is required.

**Kernel must be `Python (tau)` (this project's `.venv`)** — the first cell below prints the interpreter in use so you can confirm before going further.

## A. Setup and paths

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "tau.db"

print("PYTHON:", sys.executable)
print("PROJECT ROOT:", project_root)
print("DB PATH:", db_path, db_path.exists())


PYTHON: /Users/akshay/tau/.venv/bin/python
PROJECT ROOT: /Users/akshay/tau
DB PATH: /Users/akshay/tau/data/tau.db True


## B. Load `.env`

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(project_root / ".env")

VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
PG_DSN = os.environ.get("PG_DSN", "dbname=tau")

print("VOYAGE_API_KEY loaded:", bool(VOYAGE_API_KEY))
print("PG_DSN:", PG_DSN)


VOYAGE_API_KEY loaded: True
PG_DSN: dbname=tau


## C. Load SQLite documents

In [3]:
import sqlite3
from datetime import datetime

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

rows = conn.execute(
    """
    SELECT id, title, text, source, published_at, ingested_at, url, metadata
    FROM documents
    """
).fetchall()

documents = [dict(row) for row in rows]

for doc in documents:
    doc["published_at"] = (
        datetime.fromisoformat(doc["published_at"]) if doc["published_at"] else None
    )
    doc["ingested_at"] = datetime.fromisoformat(doc["ingested_at"])

conn.close()

print("Loaded documents:", len(documents))
documents[0]


Loaded documents: 114


{'id': 'e6cc2a2222e0bf0cf8820968c8942a6a0d77d3e1c13ef1cfc1a2fba3c02dfe42',
 'title': 'Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out',
 'text': 'When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI models in the first place.',
 'source': 'wired',
 'published_at': datetime.datetime(2026, 8, 15, 9, 0, tzinfo=datetime.timezone.utc),
 'ingested_at': datetime.datetime(2026, 8, 17, 23, 14, 59, 966738, tzinfo=datetime.timezone.utc),
 'url': 'https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/',
 'metadata': '{"feed": "wired_ai", "author": "Fernanda Gonz\\u00e1lez"}'}

## D. Build embedding texts

Text = title + body, matching the pipeline design.

In [4]:
embed_texts = [
    f'{doc["title"] or ""}\n\n{doc["text"] or ""}'.strip()
    for doc in documents
]

print(embed_texts[0][:200])


Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out

When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI mo


## E. Voyage embedding helper

In [5]:
from tau.retrieval.embeddings import embed_documents, embed_query

document_vectors = embed_documents(embed_texts, VOYAGE_API_KEY)

print("Embedded documents:", len(document_vectors), "dims:", len(document_vectors[0]))


Embedded documents: 114 dims: 1024


## F. Connect to Postgres + register pgvector

In [6]:
import psycopg
from pgvector.psycopg import register_vector
from pgvector import Vector

pg_conn = psycopg.connect(PG_DSN, autocommit=True)
pg_conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(pg_conn)

print("Connected to Postgres:", pg_conn.info.dbname)


Connected to Postgres: tau


## G. Create/populate Postgres documents table

In [7]:
pg_conn.execute(
    """
    CREATE TABLE IF NOT EXISTS documents (
        id TEXT PRIMARY KEY,
        title TEXT,
        text TEXT NOT NULL,
        source TEXT NOT NULL,
        published_at TIMESTAMPTZ,
        ingested_at TIMESTAMPTZ NOT NULL,
        url TEXT,
        metadata JSONB,
        embedding VECTOR(1024)
    )
    """
)

for doc, vector in zip(documents, document_vectors):
    pg_conn.execute(
        """
        INSERT INTO documents (
            id, title, text, source, published_at, ingested_at, url, metadata, embedding
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (id) DO UPDATE SET
            title = EXCLUDED.title,
            text = EXCLUDED.text,
            source = EXCLUDED.source,
            published_at = EXCLUDED.published_at,
            ingested_at = EXCLUDED.ingested_at,
            url = EXCLUDED.url,
            metadata = EXCLUDED.metadata,
            embedding = EXCLUDED.embedding
        """,
        (
            doc["id"],
            doc["title"],
            doc["text"],
            doc["source"],
            doc["published_at"],
            doc["ingested_at"],
            doc["url"],
            doc["metadata"],
            Vector(vector),
        ),
    )

row_count = pg_conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print("Postgres documents table row count:", row_count)


Postgres documents table row count: 114


## H. Semantic retrieval function

In [8]:
from tau.retrieval.search import semantic_search


## I. Voyage reranking

In [9]:
from tau.retrieval.search import rerank_results


## J. Temporal decay

In [10]:
from tau.retrieval.temporal import apply_temporal_decay


## K. End-to-end example query

In [11]:
query = "OpenAI safety problems"
query_vector = embed_query(query, VOYAGE_API_KEY)

results = semantic_search(pg_conn, query_vector, limit=20)
reranked_results = rerank_results(query, results, VOYAGE_API_KEY, top_k=5)
tau_results = apply_temporal_decay(reranked_results, tau_hours=24)

for r in tau_results:
    print(f'{r["title"]!r}')
    print(f'  source={r["source"]}  published_at={r["published_at"]}')
    print(
        f'  rerank_score={r["rerank_score"]:.4f}  '
        f'recency_weight={r["recency_weight"]:.4f}  '
        f'final_score={r["final_score"]:.4f}'
    )
    print()


'OpenAI lays out new security changes after its AI hacked Hugging Face'
  source=hackernews  published_at=2026-08-18 20:07:06-04:00
  rerank_score=0.6406  recency_weight=0.9551  final_score=0.6119

'OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
  source=wired  published_at=2026-08-18 14:33:11-04:00
  rerank_score=0.7383  recency_weight=0.7575  final_score=0.5592

"OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
  source=hackernews  published_at=2026-08-18 20:14:58-04:00
  rerank_score=0.4434  recency_weight=0.9604  final_score=0.4258

'The Powerful Chinese AI Model Experts Warned About Is Here'
  source=wired  published_at=2026-08-18 05:00:00-04:00
  rerank_score=0.4277  recency_weight=0.5087  final_score=0.2176

'The Safety Reckoning Inside OpenAI'
  source=wired  published_at=2026-08-13 18:37:19-04:00
  rerank_score=0.7695  recency_weight=0.0060  final_score=0.0047



## L. Temporal intent routing

`router.py` classifies a query into one of three intents and now wires that classification
straight into the retrieval pipeline via `retrieve_with_temporal_routing`:

- **`explicit_temporal`** (e.g. "last 3 hours") → hard timestamp filter → semantic search inside
  the window → rerank. No soft decay — the hard filter *is* the temporal constraint.
- **`current`** (e.g. "what's going on with X") → semantic search → rerank → soft τ decay.
- **`topical`** (e.g. "explain X's approach") → semantic search → rerank. No temporal adjustment.

The function returns `results` holding whichever of `reranked_results` / `tau_results` applies
for that route, plus the `intent` and `time_window` (if any) it used.

In [12]:
from tau.retrieval.router import retrieve_with_temporal_routing

routing_examples = [
    "What happened with OpenAI in the last 3 hours?",  # explicit_temporal
    "What's going on with OpenAI?",                     # current
    "Explain OpenAI's safety approach",                 # topical
]

routing_checks = {}

for example_query in routing_examples:
    routed = retrieve_with_temporal_routing(
        query=example_query,
        pg_conn=pg_conn,
        voyage_api_key=VOYAGE_API_KEY,
        limit=20,
        top_k=5,
        tau_hours=24,
    )

    routing_checks[routed["intent"]] = routed

    print(f'query={routed["query"]!r}')
    print(f'  intent={routed["intent"]}  time_window={routed["time_window"]}')

    for r in routed["results"]:
        print(f'  - title={r["title"]!r}')
        print(f'    source={r["source"]}  published_at={r["published_at"]}')

        score_line = f'    rerank_score={r["rerank_score"]:.4f}'
        if "final_score" in r:
            score_line += (
                f'  recency_weight={r["recency_weight"]:.4f}'
                f'  final_score={r["final_score"]:.4f}'
            )
        print(score_line)

    print()


query='What happened with OpenAI in the last 3 hours?'
  intent=explicit_temporal  time_window=(datetime.datetime(2026, 8, 18, 22, 13, 12, 916892, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 8, 19, 1, 13, 12, 916892, tzinfo=datetime.timezone.utc))
  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.6797
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.4805
  - title='Could End the RAM Crisis [video]'
    source=hackernews  published_at=2026-08-18 20:15:19-04:00
    rerank_score=0.3320
  - title='Cerebras CS-4 rack systems juice chips for every last drop of AI performance'
    source=hackernews  published_at=2026-08-18 20:14:25-04:00
    rerank_score=0.3262
  - title='Progress happens between practice sessions'
    source=hackernews  published_

query="What's going on with OpenAI?"
  intent=current  time_window=None
  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.7422  recency_weight=0.9545  final_score=0.7084
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.5977  recency_weight=0.9597  final_score=0.5736
  - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
    source=wired  published_at=2026-08-18 14:33:11-04:00
    rerank_score=0.7266  recency_weight=0.7569  final_score=0.5499
  - title='The Safety Reckoning Inside OpenAI'
    source=wired  published_at=2026-08-13 18:37:19-04:00
    rerank_score=0.6719  recency_weight=0.0060  final_score=0.0041
  - title='The White House Is Going to Expand Its AI Policy'
    source=wired  published_at=2026-08-12 17:00:00-04:00
    r

query="Explain OpenAI's safety approach"
  intent=topical  time_window=None
  - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
    source=wired  published_at=2026-08-18 14:33:11-04:00
    rerank_score=0.6875
  - title='The Safety Reckoning Inside OpenAI'
    source=wired  published_at=2026-08-13 18:37:19-04:00
    rerank_score=0.6250
  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.4980
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.4590
  - title='The White House Is Going to Expand Its AI Policy'
    source=wired  published_at=2026-08-12 17:00:00-04:00
    rerank_score=0.3711



## M. Route verification assertions

Checks, per route, that the pipeline actually behaved as intended:

- **`explicit_temporal`** — every returned `published_at` is inside `[start_time, end_time]`,
  and no tau decay fields (`recency_weight` / `final_score`) are present.
- **`current`** — every result carries `final_score` (and `recency_weight`) from tau decay.
- **`topical`** — no tau decay fields are present (`final_score` is not required).

In [13]:
# --- explicit_temporal: hard window + no tau decay --------------------------
explicit_routed = routing_checks["explicit_temporal"]
start_time, end_time = explicit_routed["time_window"]

assert explicit_routed["results"], "explicit_temporal returned no results to check"

for r in explicit_routed["results"]:
    assert r["published_at"] is not None, f'{r["title"]!r} has no published_at'
    assert start_time <= r["published_at"] <= end_time, (
        f'{r["title"]!r} published_at={r["published_at"]} outside window '
        f'[{start_time}, {end_time}]'
    )
    assert "recency_weight" not in r and "final_score" not in r, (
        f'{r["title"]!r} has tau-decay fields; explicit_temporal must not apply decay'
    )

print("PASS: explicit_temporal — every published_at is inside the extracted window")
print("PASS: explicit_temporal — no tau decay fields present")

# --- current: tau decay applied ---------------------------------------------
current_routed = routing_checks["current"]
assert current_routed["results"], "current returned no results to check"

for r in current_routed["results"]:
    assert "final_score" in r and "recency_weight" in r, (
        f'{r["title"]!r} is missing final_score/recency_weight for the current route'
    )

print("PASS: current — every result carries recency_weight/final_score")

# --- topical: no tau decay fields required -----------------------------------
topical_routed = routing_checks["topical"]
assert topical_routed["results"], "topical returned no results to check"

for r in topical_routed["results"]:
    assert "final_score" not in r and "recency_weight" not in r, (
        f'{r["title"]!r} unexpectedly carries tau-decay fields for the topical route'
    )

print("PASS: topical — no final_score/recency_weight required or present")

print()
print("All routing assertions passed.")


PASS: explicit_temporal — every published_at is inside the extracted window
PASS: explicit_temporal — no tau decay fields present
PASS: current — every result carries recency_weight/final_score
PASS: topical — no final_score/recency_weight required or present

All routing assertions passed.
